# Make datasets base

In [6]:
import random
import numpy as np
from datasets import load_dataset,Dataset,DatasetDict

### MMLU

In [2]:
ds = load_dataset('data/datasets/web/mmlu','all')
ds = DatasetDict({
    'train': ds['auxiliary_train']  # Assign the 'test' split to 'train'
})
ds['train'] = ds['train'].add_column('question_id',range(len(ds['train'])))

In [3]:
ds_homogeneous = ds['train'].train_test_split(test_size=0.1,seed=0)

print(f"Train set size: {len(ds_homogeneous['train'])}")
print(f"Test set size: {len(ds_homogeneous['test'])}")
ds_homogeneous

Train set size: 89857
Test set size: 9985


DatasetDict({
    train: Dataset({
        features: ['question', 'subject', 'choices', 'answer', 'question_id'],
        num_rows: 89857
    })
    test: Dataset({
        features: ['question', 'subject', 'choices', 'answer', 'question_id'],
        num_rows: 9985
    })
})

In [4]:
ds_homogeneous.save_to_disk('data/datasets/processed/mmlu4_homogeneous')

Saving the dataset (0/1 shards):   0%|          | 0/89857 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/9985 [00:00<?, ? examples/s]

### MMLU-PRO

#### MMLU-PRO4

In [5]:
# Load the dataset
ds = load_dataset('data/datasets/web/MMLU-Pro', 'default')

# Define the function to filter options
def filter_options(question):
    options = question['options']
    answer_index = question['answer_index']
    
    # Ensure the correct answer is included
    correct_option = options[answer_index]
    
    # Filter out 'N/A' options first
    filtered_options = [opt for opt in options if opt != 'N/A']
    
    # Add back 'N/A' options if there are less than 4 options left
    if len(filtered_options) < 4:
        filtered_options += [opt for opt in options if opt == 'N/A']
    
    # Ensure the correct answer is included
    if correct_option not in filtered_options:
        filtered_options.append(correct_option)
    
    # Limit to exactly 4 options
    filtered_options = filtered_options[:4]
    
    # Ensure the correct answer is included in the final 4
    if correct_option not in filtered_options:
        # Replace the last option if the correct one is not included
        filtered_options[-1] = correct_option

    return filtered_options, correct_option

# Process the 'test' split
def process_test4(question):
    # Filter options and create 'choices'
    question['choices'],correct_option = filter_options(question)

    # Randomize the choices
    random.shuffle(question['choices'])
    
    # Update the answer_index to match the new position of the correct answer
    question['answer'] = question['choices'].index(correct_option)
    
    # Remove irrelevant fields
    question.pop('answer_index')
    question.pop('cot_content')
    question.pop('src')
    question.pop('options')
    return question

#Seeds for reproducibility (the process is deterministic but I set this here to avoid future reproducibility issues if I make it non deterministic in future updates)
np.random.seed(42)
random.seed(42)

# Apply the processing function to the 'test' split
ds['test'] = ds['test'].map(process_test4)

# Rename the 'test' split to 'train'
ds4 = DatasetDict({
    'train': ds['test']  # Assign the 'test' split to 'train'
})

In [6]:
ds4_homogeneous = ds4['train'].train_test_split(test_size=0.1,seed=0)
print(f"Train set size: {len(ds4_homogeneous['train'])}")
print(f"Test set size: {len(ds4_homogeneous['test'])}")
ds4_homogeneous

Train set size: 10828
Test set size: 1204


DatasetDict({
    train: Dataset({
        features: ['question_id', 'question', 'answer', 'category', 'choices'],
        num_rows: 10828
    })
    test: Dataset({
        features: ['question_id', 'question', 'answer', 'category', 'choices'],
        num_rows: 1204
    })
})

In [7]:
ds4_homogeneous.save_to_disk('data/datasets/processed/mmlupro4_homogeneous')

Saving the dataset (0/1 shards):   0%|          | 0/10828 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1204 [00:00<?, ? examples/s]

In [8]:
# Define the categories for the train set
train_categories = set(['math', 'physics', 'chemistry', 'law', 'engineering', 'other', 'economics', 'health', 'psychology', 'biology', 'philosophy', 'computer science'])

# Split the dataset based on categories
def split_by_category(dataset, train_cats):
    train_indices = []
    test_indices = []
    
    for i, example in enumerate(dataset):
        if example['category'] in train_cats:
            train_indices.append(i)
        else:
            test_indices.append(i)
    
    return train_indices, test_indices

# Apply the split to your train set
train_indices, test_indices = split_by_category(ds4['train'], train_categories)

# Create the new train and test sets
new_train = ds4['train'].select(train_indices)
new_test = ds4['train'].select(test_indices)

# Update the dataset dictionary
ds4['train'] = new_train
ds4['test'] = new_test

print(f"Train set size: {len(ds4['train'])}")
print(f"Test set size: {len(ds4['test'])}")

Train set size: 10862
Test set size: 1170


In [9]:
ds4.save_to_disk('data/datasets/processed/mmlupro4_categoriesbalanced')

Saving the dataset (0/1 shards):   0%|          | 0/10862 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1170 [00:00<?, ? examples/s]

#### MMLU-PRO 10

In [10]:
# Load the dataset
ds = load_dataset('data/datasets/web/MMLU-Pro', 'default')

# Process the 'test' split
def process_test10(question):
    question['choices'] = question['options']
    correct_option = question['choices'][question['answer_index']]
    
    # Randomize the choices
    random.shuffle(question['choices'])

    # Update the answer_index to match the new position of the correct answer
    question['answer'] = question['choices'].index(correct_option)
    
    # Remove irrelevant fields
    question.pop('answer_index')
    question.pop('cot_content')
    question.pop('src')
    question.pop('options')
    return question

#Seeds for reproducibility (the process is deterministic but I set this here to avoid future reproducibility issues if I make it non deterministic in future updates)
np.random.seed(42)
random.seed(42)

ds['test'] = ds['test'].map(process_test10)

# Rename the 'test' split to 'train'
ds10 = DatasetDict({
    'train': ds['test']  # Assign the 'test' split to 'train'
})

In [11]:
ds10_homogeneous = ds10['train'].train_test_split(test_size=0.1,seed=0)
print(f"Train set size: {len(ds10_homogeneous['train'])}")
print(f"Test set size: {len(ds10_homogeneous['test'])}")
ds10_homogeneous

Train set size: 10828
Test set size: 1204


DatasetDict({
    train: Dataset({
        features: ['question_id', 'question', 'answer', 'category', 'choices'],
        num_rows: 10828
    })
    test: Dataset({
        features: ['question_id', 'question', 'answer', 'category', 'choices'],
        num_rows: 1204
    })
})

In [12]:
ds10_homogeneous.save_to_disk('data/datasets/processed/mmlupro10_homogeneous')

Saving the dataset (0/1 shards):   0%|          | 0/10828 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1204 [00:00<?, ? examples/s]

In [13]:
# Define the categories for the train set
train_categories = set(['math', 'physics', 'chemistry', 'law', 'engineering', 'other', 'economics', 'health', 'psychology', 'biology', 'philosophy', 'computer science'])

# Split the dataset based on categories
def split_by_category(dataset, train_cats):
    train_indices = []
    test_indices = []
    
    for i, example in enumerate(dataset):
        if example['category'] in train_cats:
            train_indices.append(i)
        else:
            test_indices.append(i)
    
    return train_indices, test_indices

# Apply the split to your train set
train_indices, test_indices = split_by_category(ds10['train'], train_categories)

# Create the new train and test sets
new_train = ds10['train'].select(train_indices)
new_test = ds10['train'].select(test_indices)

# Update the dataset dictionary
ds10['train'] = new_train
ds10['test'] = new_test

print(f"Train set size: {len(ds10['train'])}")
print(f"Test set size: {len(ds10['test'])}")

Train set size: 10862
Test set size: 1170


In [14]:
ds10.save_to_disk('data/datasets/processed/mmlupro10_categoriesbalanced')

Saving the dataset (0/1 shards):   0%|          | 0/10862 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1170 [00:00<?, ? examples/s]

### MED-MCQA

In [15]:
ds = load_dataset('data/datasets/web/med_mcqa','default')

def format_questions(line):
    line['choices'] = [line['opa'],line['opb'],line['opc'],line['opd']]
    line["category"] = line['subject_name']
    line['answer'] = line['cop']-1

    line["question_id"] = line["id"]
    
    line.pop("exp")
    line.pop("cop")
    line.pop("opa")
    line.pop("opb")
    line.pop("opc")
    line.pop("opd")
    line.pop("subject_name")
    line.pop("topic_name")
    line.pop("id")
    
    return line

ds = DatasetDict({
    'train': ds['train'].map(format_questions).filter(lambda line: line['choice_type']=='single')  # Assign the 'test' split to 'train'
}) 

Repo card metadata block was not found. Setting CardData to empty.


In [16]:
ds_homogeneous = ds['train'].train_test_split(test_size=0.1,seed=0)
print(f"Train set size: {len(ds_homogeneous['train'])}")
print(f"Test set size: {len(ds_homogeneous['test'])}")
ds_homogeneous

Train set size: 108688
Test set size: 12077


DatasetDict({
    train: Dataset({
        features: ['question', 'choice_type', 'choices', 'category', 'answer', 'question_id'],
        num_rows: 108688
    })
    test: Dataset({
        features: ['question', 'choice_type', 'choices', 'category', 'answer', 'question_id'],
        num_rows: 12077
    })
})

In [17]:
ds_homogeneous.save_to_disk('data/datasets/processed/medmcqa4_homogeneous')

Saving the dataset (0/1 shards):   0%|          | 0/108688 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/12077 [00:00<?, ? examples/s]

In [18]:
# Define the categories for the train set
train_categories = set(['Medicine', 'Surgery', 'Pathology', 'Pharmacology', 'Anatomy', 'Microbiology', 'Gynaecology & Obstetrics', 'Physiology', 'Biochemistry', 'Dental', 'Pediatrics', 'Ophthalmology', 'Forensic Medicine', 'ENT', 'Radiology', 'Orthopaedics', 'Anaesthesia', 'Unknown'])

# Split the dataset based on categories
def split_by_category(dataset, train_cats):
    train_indices = []
    test_indices = []
    
    for i, example in enumerate(dataset):
        if example['category'] in train_cats:
            train_indices.append(i)
        elif example['category'] == 'Unkwown':
            continue
        else:
            test_indices.append(i)
    
    return train_indices, test_indices

# Apply the split to your train set
train_indices, test_indices = split_by_category(ds['train'], train_categories)

# Create the new train and test sets
new_train = ds['train'].select(train_indices)
new_test = ds['train'].select(test_indices)

# Update the dataset dictionary
ds['train'] = new_train
ds['test'] = new_test

print(f"Train set size: {len(ds['train'])}")
print(f"Test set size: {len(ds['test'])}")
ds.save_to_disk('data/datasets/processed/medmcqa4_categoriesbalanced')

Train set size: 108684
Test set size: 12081


Saving the dataset (0/1 shards):   0%|          | 0/108684 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/12081 [00:00<?, ? examples/s]

### Math

In [24]:
import random
import numpy as np
import tqdm
from datasets import Dataset

N_questions = 120_000
P_easy = 0.3
P_medium = 0.4
P_hard = 0.3


def generate_distractors(answer, nb_dist, flip_cap=4):
    str_answer = str(answer)
    L = len(str_answer)
    middle = list(range(L)) if L <= 2 else list(range(1, L - 1))  # keep first/last digit fixed
    n = min(int(np.random.poisson(0.5*L-0.5)) + 1, flip_cap, len(middle))
    positions = np.random.choice(middle, size=n, replace=False)  # same positions for every option
    answer_combo = "".join(str_answer[p] for p in positions)

    distractors = []
    seen = {answer_combo}  # forbid the answer's own combo
    while len(distractors) < nb_dist:
        combo = "".join(str(np.random.randint(0, 10)) for _ in positions)
        if combo in seen:
            continue
        seen.add(combo)
        digits = list(str_answer)
        for p, d in zip(positions, combo):
            digits[p] = d
        distractors.append(int("".join(digits)))
    return distractors


def sample_pairs(i_low, i_high, j_low, j_high, size):
    """Sample ``size`` unique (i, j) pairs without materialising the full grid."""
    n_i = i_high - i_low
    n_j = j_high - j_low
    population = n_i * n_j
    if size > population:
        raise ValueError(f"Requested {size} unique pairs but only {population} exist.")
    flat = np.fromiter(random.sample(range(population), size), dtype=np.int64)
    i_vals = i_low + flat // n_j
    j_vals = j_low + flat % n_j
    return np.stack([i_vals, j_vals], axis=1)


def make_dataset(nb_options, seed=None):
    if seed is not None:
        np.random.seed(seed)
        random.seed(seed)

    n_easy = int(N_questions * P_easy)
    n_medium = int(N_questions * P_medium)
    n_hard = int(N_questions * P_hard)

    pairs = {
        "easy": sample_pairs(1, 10, 10, 100_000, n_easy),
        "medium": sample_pairs(10, 100, 100, 100_000, n_medium),
        "hard": sample_pairs(100, 1_000, 1_000, 100_000, n_hard),
    }

    questions = []
    answers = []
    choicess = []
    categories = []

    ops = ["+", "-", "*"]
    for level, p in pairs.items():
        print(level)
        for e, (i, j) in enumerate(tqdm.tqdm(p)):
            i, j = int(i), int(j)
            op = ops[e % 3]
            question = f"What is {j} {op} {i}?"
            category = level + op

            if op == "+":
                result = j + i
            elif op == "-":
                result = j - i
            elif op == "*":
                result = i * j
            else:
                raise ValueError(f"undefined operator: {op}")

            distractors = generate_distractors(result, nb_options - 1)
            choices = list(distractors)
            choices.append(int(result))
            random.shuffle(choices)
            answer = choices.index(int(result))

            questions.append(question)
            answers.append(answer)
            choicess.append(choices)
            categories.append(category)

    dataset = Dataset.from_dict(
        {
            "question": questions,
            "choices": [[str(c) for c in r] for r in choicess],
            "answer": answers,
            "category": categories,
        }
    )
    dataset = dataset.add_column("question_id", list(range(len(dataset))))
    return dataset


def make_free(dataset):
    """Convert a multiple-choice dataset into free-response form."""
    def _make_free(example):
        index_answer = example["answer"]
        example["answer"] = example["choices"][index_answer]
        return example

    return dataset.map(_make_free, remove_columns=["choices"])

In [25]:
math4 = make_dataset(4,seed=42)
ds_homogeneous = math4.train_test_split(test_size=0.1,seed=0)

print(f"Train set size: {len(ds_homogeneous['train'])}")
print(f"Test set size: {len(ds_homogeneous['test'])}")
ds_homogeneous

easy


100%|██████████| 36000/36000 [00:02<00:00, 14501.13it/s]


medium


100%|██████████| 48000/48000 [00:03<00:00, 15895.43it/s]


hard


100%|██████████| 36000/36000 [00:02<00:00, 14707.07it/s]


Train set size: 108000
Test set size: 12000


DatasetDict({
    train: Dataset({
        features: ['question', 'choices', 'answer', 'category', 'question_id'],
        num_rows: 108000
    })
    test: Dataset({
        features: ['question', 'choices', 'answer', 'category', 'question_id'],
        num_rows: 12000
    })
})

In [26]:
ds_homogeneous.save_to_disk('data/datasets/processed/math4_homogeneous')

Saving the dataset (0/1 shards):   0%|          | 0/108000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/12000 [00:00<?, ? examples/s]

In [27]:
math10 = make_dataset(10,seed=42)
ds_homogeneous = math10.train_test_split(test_size=0.1,seed=0)

print(f"Train set size: {len(ds_homogeneous['train'])}")
print(f"Test set size: {len(ds_homogeneous['test'])}")
ds_homogeneous

easy


100%|██████████| 36000/36000 [00:05<00:00, 7116.18it/s]


medium


100%|██████████| 48000/48000 [00:08<00:00, 5588.61it/s]


hard


100%|██████████| 36000/36000 [00:06<00:00, 5204.80it/s]


Train set size: 108000
Test set size: 12000


DatasetDict({
    train: Dataset({
        features: ['question', 'choices', 'answer', 'category', 'question_id'],
        num_rows: 108000
    })
    test: Dataset({
        features: ['question', 'choices', 'answer', 'category', 'question_id'],
        num_rows: 12000
    })
})

In [28]:
ds_homogeneous.save_to_disk('data/datasets/processed/math10_homogeneous')

Saving the dataset (0/1 shards):   0%|          | 0/108000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/12000 [00:00<?, ? examples/s]